# The goals of this notebook:

The goal of this notebook is to add FIPS codes for each NOAA event, based on the path of the event (given by begin/end latitude and longitude) when available, or identification of the NWS Public Forecast Zone within each county.

A list of FIPS codes can be found at: https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt or https://datahub.transportation.gov/Railroads/State-County-and-City-FIPS-Reference-Table/eek5-pv8d/data_preview

To identify counties based on the (linear) path of an event, we can use a high-resolution shapefile of US counties from the US Census, available from https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html.

To identify NWS zones we can use a Zone-county Correlation File from https://www.weather.gov/gis/ZoneCounty (we could also try to use a shapefile from the National Weather Service, available from https://www.weather.gov/gis/PublicZones, but these zone numbers don't appear to match what are in the NOAA file).

We will then also add variables to the NOAA data to reflect the severity of the event. This includes:
- The overall duration of the event
- Predictors that are already in the NOAA data
- Maybe some ERA5 data, if we can figure out how to merge

Then, we'll create a new version of the data in which each event that occurred in multiple counties is split into a single line for the event in each county. We'll add a "duration per county" variable.

Finally, we'll convert these data into a time series based on the start and endtimes of each event.

First, load the NOAA data and then convert the state names to abbreviations

In [78]:
import pandas as pd

df_events = pd.read_csv("../Data/NOAA_StormEvents/StormEvents_2014_2024.csv")

In [79]:
# We'll use a dictionary to do the conversion.
# (There is a Pandas 'us' package that could do this for us, but I'm having dependency issues, so a dictionary it is...)

us_state_to_abbrev = {
    "ALABAMA": "AL",
    "ALASKA": "AK",
    "ARIZONA": "AZ",
    "ARKANSAS": "AR",
    "CALIFORNIA": "CA",
    "COLORADO": "CO",
    "CONNECTICUT": "CT",
    "DELAWARE": "DE",
    "FLORIDA": "FL",
    "GEORGIA": "GA",
    "HAWAII": "HI",
    "IDAHO": "ID",
    "ILLINOIS": "IL",
    "INDIANA": "IN",
    "IOWA": "IA",
    "KANSAS": "KS",
    "KENTUCKY": "KY",
    "LOUISIANA": "LA",
    "MAINE": "ME",
    "MARYLAND": "MD",
    "MASSACHUSETTS": "MA",
    "MICHIGAN": "MI",
    "MINNESOTA": "MN",
    "MISSISSIPPI": "MS",
    "MISSOURI": "MO",
    "MONTANA": "MT",
    "NEBRASKA": "NE",
    "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH",
    "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM",
    "NEW YORK": "NY",
    "NORTH CAROLINA": "NC",
    "NORTH DAKOTA": "ND",
    "OHIO": "OH",
    "OKLAHOMA": "OK",
    "OREGON": "OR",
    "PENNSYLVANIA": "PA",
    "RHODE ISLAND": "RI",
    "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD",
    "TENNESSEE": "TN",
    "TEXAS": "TX",
    "UTAH": "UT",
    "VERMONT": "VT",
    "VIRGINIA": "VA",
    "WASHINGTON": "WA",
    "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI",
    "WYOMING": "WY",
    "DISTRICT OF COLUMBIA": "DC",
    "AMERICAN SAMOA": "AS",
    "GUAM": "GU",
    "NORTHERN MARIANA ISLANDS": "MP",
    "PUERTO RICO": "PR",
    "UNITED STATES MINOR OUTLYING ISLANDS": "UM",
    "U.S": "US"
}

def convert_state_name_to_abbrev(state_name):
    #Note that the state name needs to be in all capital letters
    return us_state_to_abbrev.get(state_name, "Unknown")

# Apply the function to convert the STATE variable in df_events to abbreviations
df_events['STATE_ABBREV'] = df_events['STATE'].apply(convert_state_name_to_abbrev)

Many (about 16,000) of these observations are from bodies of water/oceans or from US territories or protectorates. We'll drop these data.

In [101]:
#Drop rows where STATE_ABBREV is either Unknown, AS, GU, MP, PR, UM, or US
df_events = df_events[~df_events['STATE_ABBREV'].isin(["Unknown", "AS", "GU", "MP", "PR", "UM", "US"])]

Next, load the US Counties shapefile from the US Census.

In [80]:
import geopandas

#Load the US Census Counties shapefile
counties = geopandas.read_file('../Data/cb_2023_us_county_500k')

#Concatenate STATEFP and COUNTFP and then convert to an integer
counties['FIPS'] = (counties['STATEFP'].astype(str) + counties['COUNTYFP'].astype(str)).astype(int)

Define a function that identifies all the FIPS in the path of the weather event.

This takes about 2 minutes to run.

We're going to assume that the path of the event is essentially linear. This is probably inaccurate, but we don't have any additional data (without doing something really clever with the ERA5 data) as an alternative

In [81]:
# Given a beginning point (given by BEGIN_LAT and BEGIN_LON) and an ending point (given by END_LAT and END_LON) from df_events, 
# identify which FIPS values from counties lie in the path between the beginning and ending points

def get_fips_from_path(row):
    #Get the beginning and ending values of longitude and latitude
    begin = (row['BEGIN_LON'], row['BEGIN_LAT'])
    end = (row['END_LON'], row['END_LAT'])

    #Convert them into geopandas points
    points = geopandas.points_from_xy([begin[0], end[0]], [begin[1], end[1]])

    #Get the path between the two points
    path = geopandas.GeoSeries(points)
    
    #Get the FIPS values from the counties shapefile that intersect with the path
    fips = counties[counties.geometry.intersects(path.union_all())]['FIPS'].tolist()
    
    return fips

#Apply get_fips_from_path to each row of df_events and create a new variable that lists the fips values
df_events['FIPS_from_Path'] = df_events.apply(get_fips_from_path, axis=1)

The NOAA data includes columns for STATE_FIPS and CZ_FIPS. When the CZ_TYPE variable is C (i.e., County/Parish), these values appear to match "official lists."

However, when CZ_TYPE is Z, these appear to refer to a NWS forecast zone. These zones often correspond to counties, but can also correspond to sub-areas of counties 
that might experience different weather patterns (e.g., a particularly tall mountain).

The NWS has a zone-county correlation file that we can use to look up the FIPS code based on a concatenation of the state abbreviation and the CZ_FIPS value for *most* of the values. For others, we'll need to resort to some other scraping.

In [ ]:
#Import the zone-county correlation file
df_zonecountycorr = pd.read_csv('../Data/bp05mr24.dbx', sep="|")

#Add the column headings
df_zonecountycorr.columns = ['STATE', 'ZONE', 'CWA', 'NAME', 'STATE_ZONE', 'COUNTY', 'FIPS', 'TIME_ZONE', 'FE_AREA', 'LAT', 'LON']

#Create a new variable in df_events by first adding zeroes to make CZ_FIPS a three-digit number and then concatenating with STATE_ABBREV
df_events['CZ_FIPS_ZONE'] = df_events['STATE_ABBREV'] + df_events['CZ_FIPS'].apply(lambda x: str(x).zfill(3))

#Merge the FIPS column from df_zonecountycorr with df_events, matching on CZ_FIPS_ZONE from df_events and STATE_ZONE from df_zonecountycorr
df_events = df_events.merge(df_zonecountycorr[['STATE_ZONE', 'FIPS']], left_on='CZ_FIPS_ZONE', right_on='STATE_ZONE', how='left')

#Rename the FIPS column FIPS_from_Zone
df_events.rename(columns={'FIPS': 'FIPS_from_Zone'}, inplace=True)

#Drop the CZ_FIPS_ZONE and STATE_ZONE variables from df_events
df_events.drop(columns=['CZ_FIPS_ZONE', 'STATE_ZONE'], inplace=True)

There are still some FIPS codes that aren't being identified. We'll try to match the CZ_NAME value in df_events with the NAME variable in df_zonecountycorr using jellyfish

Note that this takes nearly 35 minutes to run...

In [89]:
#Import jellyfish
import jellyfish

#For each row in df_events, use its STATE_ABBREV to create a new data frame from matching values of the STATE variable in df_zonecountycorr
#Then use jaro_winkler to do a fuzzy match on the CZ_NAME variable in df_events with the NAME variable in the new data frame. For the best match, import the FIPS value from df_zonecountycorr

def get_fips_from_name(row):
    #Get the state abbreviation
    state_abbrev = row['STATE_ABBREV']
    
    #Create a new data frame from matching values of the STATE variable in df_zonecountycorr
    df_state = df_zonecountycorr[df_zonecountycorr['STATE'] == state_abbrev]
    
    #Get the CZ_NAME from the row
    cz_name = row['CZ_NAME']
    
    #Get the best match using jaro_winkler
    best_match = None
    best_score = 0
    best_fips = None
    
    for index, row in df_state.iterrows():
        score = jellyfish.jaro_winkler_similarity(cz_name, row['NAME'])
        if score > best_score:
            best_score = score
            best_match = row['NAME']
            best_fips = row['FIPS']
    
    return best_fips

#Apply get_fips_from_name to each row of df_events and create a new variable that lists the fips values
df_events['FIPS_from_Name'] = df_events.apply(get_fips_from_name, axis=1)

Now, we'll create a list of all the FIPS associated with each event. For instances where we know the path, we'll default to that. If we don't know the path, then we'll use FIPS_from_Zone; if we don't have either, then we'll use FIPS_from_Name

In [107]:
#to ensure that our FIPS values are all lists, we'll first transform FIPS_from_Zone and FIPS_from_Name into lists
df_events['FIPS_from_Zone'] = df_events['FIPS_from_Zone'].apply(lambda x: [x] if pd.notna(x) else [])
df_events['FIPS_from_Name'] = df_events['FIPS_from_Name'].apply(lambda x: [x] if pd.notna(x) else [])

#Then we'll create a new variable that is equal to FIPS_from_Path if that's not empty; if it is we'll use FIPS_from_Zone and then FIPS_from_Name
df_events['FIPS'] = df_events.apply(lambda x: x['FIPS_from_Path'] if len(x['FIPS_from_Path']) > 0 else (x['FIPS_from_Zone'] if pd.notna(x['FIPS_from_Zone']) else x['FIPS_from_Name']), axis=1)

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/3395774983.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events['FIPS_from_Zone'] = df_events['FIPS_from_Zone'].apply(lambda x: [x] if pd.notna(x) else [])
/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/3395774983.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events['FIPS_from_Name'] = df_events['FIPS_from_Name'].apply(lambda x: [x] if pd.notna(x) else [])
/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipy

In [109]:
#Add a new variable that is equal to the length of FIPS
df_events['Number_of_FIPS'] = df_events['FIPS'].apply(len)

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/2402520471.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events['Number_of_FIPS'] = df_events['FIPS'].apply(len)


There are a bunch of variables we don't need. We'll drop them here.

In [111]:
#Drop the variables 
# BEGIN_YEARMONTH, BEGIN_DAY, BEGIN_TIME, 
# END_YEARMONTH, END_DAY, END_TIME, 
# EPISODE_ID, EVENT_ID, 
# STATE, STATE_FIPS, 
# YEAR, MONTH_NAME, 
# CZ_TYPE, CZ_FIPS, CZ_NAME, WFO, SOURCE, 
# FLOOD_CAUSE, CATEGORY, 
# TOR_LENGTH, TOR_WIDTH, TOR_OTHER_WFO, TOR_OTHER_CZ_STATE, TOR_OTHER_CZ_FIPS, TOR_OTHER_CZ_NAME, 
# BEGIN_RANGE, BEGIN_AZIMUTH, BEGIN_LOCATION, END_RANGE, END_AZIMUTH, END_LOCATION, DATA_SOURCE

df_events.drop(columns=['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME',
                        'END_YEARMONTH', 'END_DAY', 'END_TIME',
                        'EPISODE_ID', 'EVENT_ID',
                        'STATE', 'STATE_FIPS',
                        'YEAR', 'MONTH_NAME',
                        'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'SOURCE',
                        'FLOOD_CAUSE', 'CATEGORY',
                        'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME',
                        'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'DATA_SOURCE'], inplace=True)

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/2830267991.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events.drop(columns=['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME',


There are several ways we could measure the severity of each event. One basic way is to compute the duration of the event based on the begin year/day/time (measured by the BEGIN_DATE_TIME variable) and end year/day/time (measured by the END_DATE_TIME variable). 

In general, we should be mindful of time zone differences. However, the NOAA data don't indicate which time zone the begin/end data are coming from. So the best we can do is assume that each event is confined to a single time zone, even for events that impact multiple counties.

In [113]:
#Create a new variable in df_events total_duration by subtracting the BEGIN_DATE_TIME from END_DATE_TIME, measuring in minutes.
#Specify the format day-month-year hour:minute:second
df_events['BEGIN_DATE_TIME'] = pd.to_datetime(df_events['BEGIN_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df_events['END_DATE_TIME'] = pd.to_datetime(df_events['END_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df_events['total_duration_min'] = (df_events['END_DATE_TIME'] - df_events['BEGIN_DATE_TIME']).dt.total_seconds() / 60.0

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/2944949999.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events['BEGIN_DATE_TIME'] = pd.to_datetime(df_events['BEGIN_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/2944949999.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events['END_DATE_TIME'] = pd.to_datetime(df_events['END_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipy

For events that involve multiple counties, we could assume that either (1) the event is localized enough that it only impacts one county at a time, or (2) it affects multiple counties simultaneously. If we assume #1, then we might want to divide the total duration by the number of counties in the path of the event.

In [114]:
#Divide total_duration_min by the number of items in the FIPS list
df_events['total_duration_perfips_min'] = df_events['total_duration_min'] / df_events['Number_of_FIPS']

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/2802729644.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events['total_duration_perfips_min'] = df_events['total_duration_min'] / df_events['Number_of_FIPS']


Now we'll start exporting these data. At first, we'll keep things basically as they are with one row per event, regardless of the number of counties in which the event takes place.

To enable merging, we'll associate each event with the first FIPS value in its list.

Then we'll export as a csv

In [117]:
#Create a new variable FIPS_First that is the first value in the list of each FIPS value
df_events['FIPS_First'] = df_events['FIPS'].apply(lambda x: x[0] if len(x) > 0 else None).astype(int)

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_45777/3166319170.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_events['FIPS_First'] = df_events['FIPS'].apply(lambda x: x[0] if len(x) > 0 else None).astype(int)


The next thing we'd like to do is to associate each event with a power grid subregion. This should generally be straightforward--most of these events occurred in a single county/FIPS value. But some of the larger events could span multiple counties and, potentially, multiple subregions.

In [ ]:
#Load the Counties_2020.csv file from ../Data/
df_counties_2020 = pd.read_csv('../Data/Counties_2020.csv')

#Merge df_events with the Subregion variable from df_counties using FIPS_First from df_events and FIPS from df_counties_2020
df_events = df_events.merge(df_counties_2020[['FIPS', 'Subregion']], left_on='FIPS_First', right_on='FIPS', how='left')

#Remove the variable FIPS_y
df_events.drop(columns=['FIPS_y'], inplace=True)

#Rename FIPS_x as FIPS
df_events.rename(columns={'FIPS_x': 'FIPS'}, inplace=True)

Now we can start exporting to files. First, we'll export the cleaned version without any modifications

In [126]:
#Export df_events as a parquet file
df_events.to_parquet('../Data/NOAA_Cleaned_FirstFIPS.parquet', index=False)

Next, we'll split each multi-FIPS event into separate rows. Then we can look up the eGRID subregion for the FIPS, and then export

In [ ]:
#For each row in df_events where Number_of_FIPS >1, make one copy of that row for each item in the FIPS list
df_events_exploded = df_events.explode('FIPS')

#Drop the FIPS_First and Subregion variables
df_events_exploded.drop(columns=['FIPS_First', 'Subregion'], inplace=True)

#Merge df_events with the Subregion variable from df_counties using FIPS_First from df_events and FIPS from df_counties_2020
df_events_exploded = df_events_exploded.merge(df_counties_2020[['FIPS', 'Subregion']], left_on='FIPS', right_on='FIPS', how='left')

In [136]:
#Export df_events as a parquet file
df_events_exploded.to_parquet('../Data/NOAA_Cleaned_ExplodedFIPS.parquet', index=False)

In [139]:
df_events

,EVENT_TYPE,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,...,STATE_ABBREV,FIPS_from_Path,FIPS_from_Zone,FIPS_from_Name,FIPS,Number_of_FIPS,total_duration_min,total_duration_perfips_min,FIPS_First,Subregion
0,Heavy Snow,2014-02-18 10:00:00,EST-5,2014-02-18 20:00:00,0,0,0,0,0.00K,0.00K,...,NH,[],[33011.0],[33009.0],[33011.0],1,600.0,600.0,33011,NEWE
1,Flood,2014-03-30 08:31:00,EST-5,2014-03-30 09:31:00,0,0,0,0,35.00K,0.00K,...,MA,[25017],[25005.0],[25009.0],[25017],1,60.0,60.0,25017,NEWE
2,Hail,2014-04-27 23:06:00,CST-6,2014-04-27 23:06:00,0,0,0,0,0.00K,0.00K,...,MO,[29067],[29185.0],[29065.0],[29067],1,0.0,0.0,29067,SRMW
3,Thunderstorm Wind,2014-04-27 23:03:00,CST-6,2014-04-27 23:03:00,0,0,0,0,10.00K,0.00K,...,MO,[29067],[29185.0],[29065.0],[29067],1,0.0,0.0,29067,SRMW
4,High Wind,2014-02-15 13:00:00,PST-8,2014-02-15 21:00:00,0,0,0,0,0.00K,0.00K,...,WA,[],[],[53059.0],[53059.0],1,480.0,480.0,53059,NWPP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
750216,Thunderstorm Wind,2024-05-26 11:48:00,EST-5,2024-05-26 11:48:00,0,0,0,0,NaN,NaN,...,KY,[21021],[21177.0],[21187.0],[21021],1,0.0,0.0,21021,SRTV
750217,Thunderstorm Wind,2024-05-22 18:09:00,EST-5,2024-05-22 18:09:00,0,0,0,0,0.00K,0.00K,...,IN,[18143],[],[18021.0],[18143],1,0.0,0.0,18143,RFCW
750218,Thunderstorm Wind,2024-05-22 17:57:00,EST-5,2024-05-22 17:57:00,0,0,0,0,0.00K,0.00K,...,IN,[18123],[],[18125.0],[18123],1,0.0,0.0,18123,RFCW
750219,Hail,2024-06-23 17:45:00,EST-5,2024-06-23 17:50:00,0,0,0,0,0.00K,0.00K,...,NH,[33015],[33011.0],[33009.0],[33015],1,5.0,5.0,33015,NEWE


Finally, we'll convert df_events into a time series and then export

In [ ]:
#Start by defining the event categories
snowice_types = ['Heavy Snow', 'Winter Storm', 'Winter Weather', 'Ice Storm', 'Extreme Cold/Wind Chill','Blizzard', 'Avalanche', 'Cold/Wind Chill', 'Frost/Freeze','Sleet','Freezing Fog', 'Lake-Effect Snow']
flood_types = ['Flash Flood', 'Coastal Flood','Lakeshore Flood','Debris Flow']
storm_types = ['Heavy Rain','Tropical Storm','Tropical Depression','Hail','Lightning','Marine Lightning']
hurricane_types = ['Hurricane','Hurricane (Typhoon)']
heat_types = ['Excessive Heat', 'Heat']
fire_types= ['Wildfire','Dense Smoke']
wind_types = ['High Wind', 'Strong Wind', 'Thunderstorm Wind', 'Marine Thunderstorm Wind','Tornado','Waterspout', 'Funnel Cloud']
ocean_types = ['Marine Hurricane/Typhoon','Rip Current','Astronomical Low Tide', 'Marine Dense Fog','Marine Tropical Depression','Marine Strong Wind', 'Marine High Wind', 'Marine Hail','High Surf', 'Marine Tropical Storm', 'Seiche','Storm Surge/Tide', 'Tsunami', 'Sneakerwave']
other_types = ['Drought','Dust Storm', 'Dense Fog', 'Dust Devil', 'Volcanic Ashfall']

#Create a new variable in df_events that identifies the list that EVENT_TYPE is in
df_events['EVENT_CATEGORY'] = df_events['EVENT_TYPE'].apply(lambda x: 'Snow/Ice' if x in snowice_types else ('Flood' if x in flood_types else ('Storm' if x in storm_types else ('Hurricane' if x in hurricane_types else ('Heat' if x in heat_types else ('Fire' if x in fire_types else ('Wind' if x in wind_types else ('Ocean' if x in ocean_types else ('Other' if x in other_types else 'Unknown')))))))))

event_categories = ['SnowIce', 'Flood', 'Storm', 'Hurricane', 'Heat', 'Fire', 'Wind', 'Ocean', 'Other']


# Specify the FIPS code for which we're making the dataframe (we'll loop through these)
# Note that this is stored as either an integer or a float, so codes with leading zeroes will have the leading zero dropped
fips = fipscode

# Generate the time range for the new DataFrame
start_date = datetime(2014, 1, 1)
end_date = datetime(2021, 12, 31, 18, 00)  # Include the last time interval that starts at 18:00
time_index = pd.date_range(start=start_date, end=end_date, freq='6h')

# Create a new dataframe using time_index above as one of the variables
new_df = pd.DataFrame({'time': time_index})

# Initialize event count columns for each event type
for category in event_categories:
    new_df[f'event_count {category}'] = 0  # Initialize event counts to 0

#Note that we already converted BEGIN_DATETIME AND END_DATETIME into datetime objects above



# Filter the NOAA data for the specified state and event type
filtered_df = df[
    (df['FIPS'] == fips) & 
    (df['EVENT_CATEGORY'].isin(event_categories)) & 
    (df['END_DATETIME'] >= start_date) & 
    (df['BEGIN_DATETIME'] <= end_date)
].copy(deep=True)

# Iterate through the events and assign them to the closest time interval in the new DataFrame
for event_category in event_categories:
    event_subset = filtered_df[filtered_df['EVENT_CATEGORY']==event_category]
    
    for _, row in event_subset.iterrows():
        event_start = row['BEGIN_DATETIME']
        event_end = row['END_DATETIME']
    
        # Round the start and end times to the nearest 6-hour interval
        event_start_rounded = event_start.round('6h')
        event_end_rounded = event_end.round('6h')
    
        # Find the indices in the new DataFrame for the rounded times
        start_idx = new_df['time'].searchsorted(event_start_rounded)
        end_idx = new_df['time'].searchsorted(event_end_rounded)
    
        # Increment the event count for the affected time intervals
        if start_idx < len(new_df) and end_idx <= len(new_df):
            new_df.loc[start_idx:end_idx, f'event_count {event_category}'] += 1



# Unused Code

**Note: Now that we have NWS Forecast Zone information, we're no longer using the code below. But I'm keeping it around just in case we ever want to re-run it**

There are many rows in df_events that don't have beginning/end longitude/latitude values. However, the event_narrative variable sometimes mentions the name of a county.

The code below searches for county names in the event_narrative variable and matches them to a FIPS from the counties dataframe

Note that this takes almost 5 minutes to run

In [44]:
def get_fips_from_narrative(row):
    #Get the state from the row
    state = row['STATE_ABBREV']
    
    #Get the event narrative from the row
    narrative = row['EVENT_NARRATIVE']

    #Create a new dataframe from county_fips that has the same state
    counties_state = counties[counties['STUSPS'] == state]
    
    #Identify any word in the EVENT_NARRATIVE variable that matches the COUNTYNAME variable in county_fips for the same STATE
    fips = []
    for index, row in counties_state.iterrows():
        #if narrative is not NaN:
        if pd.notna(row['NAME']) and pd.notna(narrative):
            if row['NAME'] in narrative:
                fips.append(row['FIPS'])    
    return fips

#Apply the function to each row of df_events that lack latitude data to add a FIPS code
df_events['FIPS_from_Countyname_Narrative'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_narrative, axis=1)

In addition to county names, sometimes the event_narrative variable refers to (what seem to be) various sorts of weather stations.

Like the county names, we can try to use these names to identify the FIPS where the weather event was reported.

Use the list of weather stations from Meteostat (https://github.com/meteostat/weather-stations?tab=readme-ov-file) to further identify locations

Note that lists of additional weather stations are available from NOAA: https://www.ncei.noaa.gov/access/homr/#. However, I've had trouble reliably identifying examples of weather stations from the downloaded lists, so sticking with meteostat for now.

In the code below, we'll load the Meteostat list, do some cleaning, add a geometry variable so we can merge it with the counties dataframe, and look up the FIPS code for each station from the counties data

In [45]:
#Load the Meteostat list of weather stations
import pandas as pd
df_meteostat_list = pd.read_json('../Data/full.json')

#Restrict the country variable to "US"
df_meteostat_list = df_meteostat_list[df_meteostat_list['country'] == "US"]

#Convert the name variable into strings and strip the text {'en': '
df_meteostat_list['name'] = df_meteostat_list['name'].astype(str).str.strip("{'en': '").str.strip("'").str.strip("'}")

#Extract the icao value from the identifiers variable
df_meteostat_list['icao'] = df_meteostat_list['identifiers'].apply(lambda x: x['icao'])

#Extract the latitude, longitude, and elevation from the location variable
df_meteostat_list['latitude'] = df_meteostat_list['location'].apply(lambda x: x['latitude'])
df_meteostat_list['longitude'] = df_meteostat_list['location'].apply(lambda x: x['longitude'])
df_meteostat_list['elevation'] = df_meteostat_list['location'].apply(lambda x: x['elevation'])

#Drop the identifiers, location, and inventory variables
df_meteostat_list = df_meteostat_list.drop(columns=['country','identifiers', 'location', 'inventory'])

#Convert the id, name, region, and icao variables into strings
df_meteostat_list['id'] = df_meteostat_list['id'].astype(str)
df_meteostat_list['name'] = df_meteostat_list['name'].astype(str)
df_meteostat_list['region'] = df_meteostat_list['region'].astype(str)
df_meteostat_list['icao'] = df_meteostat_list['icao'].astype(str)

#Convert df_meteostat_list into a geopandas dataframe
df_meteostat_list = geopandas.GeoDataFrame(df_meteostat_list, geometry=geopandas.points_from_xy(df_meteostat_list['longitude'], df_meteostat_list['latitude']))

#Set the CRS as EPSG 4269
# Note: I'm not entirely sure this is the CRS for the data - I couldn't find any info directly from the meteostat website
df_meteostat_list.set_crs(epsg=4269, inplace=True)

#Perform a spatial join between the counties and the df_meteostat_list dataframes to add the FIPS variable to df_meteostat_list
df_meteostat_list['FIPS'] = geopandas.sjoin(df_meteostat_list, counties, how='left', predicate='intersects')['FIPS']

Next we'll define a function that will look through the event narrative and look for mentions of weather stations, then find the corresponding FIPS code

Note that the next code chunk takes nearly 4 minutes to run.

In [46]:
def get_fips_from_meteostat(df_events_row):
    #Get the state from the row
    state = df_events_row['STATE_ABBREV']
    
    #Get the event narrative from the row
    narrative = df_events_row['EVENT_NARRATIVE']
    
    #Create a new dataframe from df_meteostat_list that has the same state
    df_meteostat_list_state = df_meteostat_list[df_meteostat_list['region'] == state]
    
    #Identify any word in the EVENT_NARRATIVE variable that matches the id variable, the icao variable, or the name variable in df_meteostat_list for the same STATE
    fips = []
    for index, row in df_meteostat_list_state.iterrows():
        if pd.notna(narrative):
            if str(row['name']) in narrative or str(row['icao']) in narrative or str(row['id']) in narrative:
                fips.append(row['FIPS'])
    return fips

#Apply the function to each row of df_events that lack latitude data and create a new variable that lists the fips values
df_events['FIPS_from_Meteostat'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_meteostat, axis=1)

The code below attempts to extract FIPS information from the CZ_NAME variable

In [47]:
def get_fips_from_czname(row):
    #Get the state from the row
    state = row['STATE_ABBREV']
    
    #Get the county name from the row
    czname = row['CZ_NAME']
    
    #Create a new dataframe from counties that has the same state
    counties_state = counties[counties['STUSPS'] == state]
    
    fips = []
    for index, row in counties_state.iterrows():
        #if COUNTYNAME is not NaN and narrative is not NaN:
        if pd.notna(czname):
            #Convert row['NAME'] to all capitals
            uppername = row['NAME'].upper()
            if uppername in czname:
                fips.append(row['FIPS'])
    #print(fips)
    
    return fips

#Apply get_fips_from_narrative to each row of df_events and create a new variable that lists the fips values
# But do this only for rows of df_events that lack latitude and/or longitude data and only for rows of df_events that have a value for STATE_ABBREV
df_events['FIPS_from_CZNAME'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_czname, axis=1)

The NOAA data also includes several measures of the severity of the impact of the event: Injuries, Deaths, and Damage

In 2023, the US DoT's official value of a statistical life is $13.2 million. (see https://www.transportation.gov/office-policy/transportation-policy/revised-departmental-guidance-on-valuation-of-a-statistical-life-in-economic-analysis)

Information about statistical injuries are trickier to identify. We have some data from Switzerland:
https://ansperformance.eu/economics/cba/standard-inputs/chapters/value_of_a_statistical_injury.html
(It looks like the value of a life in Switzerland is only $4.3 million....)

Some data from 2013 - again, from Switzerland - that an injury was valued at $35k Swiss francs. Assuming 1.14 francs per USD. With inflation (data from https://www.inflationtool.com/euro-switzerland/2013-to-present-value), this would rise to 1.21 francs per USD in 2025. So the value of an injury in 2025 USD would be $42,350.

The various types of damage are already given in thousands of dollars.

We can combine these columns to get a compsite cost for each event.

Not sure we'll actually be using this, but keeping the code around for now.

In [ ]:
#In the DAMAGE_PROPERTY and DAMAGE_CROPS variables, if the string ends with 'K', remove the 'K' at the end of the string, convert to an integer, and multiply by 1000. If the string ends with 'M', remove the 'M" at the end of the string, convert to an integer, and multiply by 1,000,000
def convert_damage(damage):
    if isinstance(damage, str):
        if damage.endswith('K'):
            return float(damage[:-1]) * 1000
        elif damage.endswith('M'):
            return float(damage[:-1]) * 1000000
    else:
        return 0

#Compute the total value of damage to property and crops
df_events['DAMAGE_PROPERTY_VALUE'] = df_events['DAMAGE_PROPERTY'].apply(convert_damage)
df_events['DAMAGE_CROPS_VALUE'] = df_events['DAMAGE_CROPS'].apply(convert_damage)

#Compute the total values of injuries and deaths
df_events['INJURIES_TOTAL_VALUE'] = (df_events['INJURIES_DIRECT'] + df_events['INJURIES_INDIRECT']) * 42350
df_events['DEATHS_TOTAL_VALUE'] = (df_events['DEATHS_DIRECT'] + df_events['DEATHS_INDIRECT']) * 13200000

#Compute the total value of damage, injuries, and deaths for each event
df_events['TOTAL_VALUE'] = df_events['DAMAGE_PROPERTY_VALUE'] + df_events['DAMAGE_CROPS_VALUE'] + df_events['INJURIES_TOTAL_VALUE'] + df_events['DEATHS_TOTAL_VALUE']

#Also create a version per FIPS
#Divide TOTAL_VALUE by Number_of_FIPS_in_Event if Number_of_FIPS_in_Event is nonzero; otherwise divide by 1
df_events['TOTAL_VALUE_PER_FIPS'] = df_events['TOTAL_VALUE'] / df_events['Number_of_FIPS_in_Event'].replace(0, 1)